In [ ]:
import os, sys, torch, glob, random, itertools
import torch.nn.functional as F
import pandas as pd, numpy as np
import plotly.graph_objects as go
sys.path.insert(0, r"c:\repos\DroneDetectionRF")
from NoisyUAV.funciones.dataset.cargador import cargar_muestra
from NoisyUAV.funciones.dsp_rf.detector_entropia import (
    detectar_bursts, plot_muestra, print_diagnostico, plot_espectrograma_3d
)
# NUEVO: Importamos el modelo híbrido y la función para crear el espectrograma
from NoisyUAV.modelo_alumn_xin.alumn_xin_cvcnn import AlumnXinCVCNN
from NoisyUAV.modelo_alumn_xin.alumn_xin_dataset import burst_iq_to_xin_tensor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
ruta_pesos = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_xin\resultados\checkpoints\alumn_xin_model_best.pt"
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model = AlumnXinCVCNN(phys_dim=8, kernel_size=5).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32).to(device)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32).to(device)
print(f"✅ Nuevo modelo ALUMN_XIN (Híbrido) cargado con éxito en {device.type.upper()}")
print(f"   Validation F1: {ckpt.get('best_val_f1', 0.0):.4f}")

In [ ]:
# PARÁMETROS
TARGET_DESEADO = 6
SNR_DESEADA    = -16
SOLO_TEST      = True
RUTA_RAW_DIR = r"C:\TFM_data\NoisyUAV\drone_RF_data"
csv_path = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_xin\alumn_xin_dataset.csv"
df_pseudo = pd.read_csv(csv_path)
filtro = (df_pseudo['target_multiclass'] == TARGET_DESEADO) & (df_pseudo['snr'] == SNR_DESEADA)
if SOLO_TEST: filtro &= (df_pseudo['split'] == 'test')
df_candidatos = df_pseudo[filtro]
if df_candidatos.empty: raise ValueError("❌ No hay ráfagas de TEST.")
# 3. Elegir un ARCHIVO aleatorio y definir variables para el gráfico
archivos_unicos = df_candidatos['file_path'].unique()
filename_random = random.choice(df_candidatos['file_path'].unique())
parent_filename = filename_random  # <--- Esta es la que te faltaba
ruta_completa = os.path.join(RUTA_RAW_DIR, filename_random)
print("=" * 60)
print(f"🎬 Muestra de TEST Recuperada: {parent_filename}")
print(f"   (Encontrados {len(archivos_unicos)} archivos de test para este caso)")
print(f"   (Este archivo contiene {len(df_candidatos[df_candidatos['file_path']==filename_random])} ráfagas en el dataset)")
print("=" * 60)
# 4. Cargar el tensor y la metadata original
# cargar_muestra devuelve: iq_tensor, label, target_id, snr_dB
iq_tensor, _, original_target, original_snr = cargar_muestra(ruta_completa)

In [ ]:
# Variables Físicas de la Tesis
FS = 14e6
NPERSEG = 2048
Z_THRESH = 0.75     
MIN_BURST_MS = 0.1
MERGE_GAP_MS = 0.75
MIN_Z_ABS = 2.0
BG_MULT = 4
MAX_BINS_FRAC = 1
SMOOTH_MS = 0.2
ADAPTIVE_WINDOW_MS = 5 

# FS = 14e6
# NPERSEG = 2048
# Z_THRESH = 1.0       # <--- SUPER RELAJADO PARA CAZAR EL FONDO DEL RUIDO
# MIN_BURST_MS = 0.25  # <--- Más corto permitido
# MERGE_GAP_MS = 0.5
# MIN_Z_ABS = 1.0      # <--- Dejamos pasar todo
# BG_MULT = 4
# MAX_BINS_FRAC = 0.25 # <--- Aquí dejamos todo, que decida el Teacher
# SMOOTH_MS = 0.2
# ADAPTIVE_WINDOW_MS = 15 

# Detección (Extracción en crudo, SIN UMBRAL DINÁMICO, para ver si el Oráculo sabe distinguir)
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target=f"Target {original_target}", snr=f"{original_snr}", index=0,
)
# Magia Visual de tu Proyecto
fig_2d = plot_muestra(
    iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
    titulo=f"Validación ORÁCULO: {parent_filename}"
)
fig_2d.show()

In [ ]:
# fig_3d = plot_espectrograma_3d(
#     iq_tensor, fs=FS, nperseg=NPERSEG, 
#     t_lim_ms=None, smooth_sigma=1.0, floor_pct=1,
#     titulo="Espectro Base (NoisyUAV 14 MHz)"
# )
# fig_3d.show()

In [ ]:
print("==================================================")
print("     VEREDICTO MODELO HÍBRIDO (Inferencia Pura)   ")
print("==================================================")
drones_encontrados, ruidos_encontrados = 0, 0
if len(bursts) == 0:
    print("  ❌ No se detectaron ráfagas.")
else:
    global_nf      = float(np.median(nf_v))
    global_ns_val  = float(np.clip(ns, 0, 5))
    global_H_mean  = float(np.mean(H_smooth))
    global_p75_act = float(np.percentile(n_active, 75))
    
    with torch.no_grad():
        for i, b in enumerate(bursts):
            # 1. RECORTAR ONDA
            idx_inicio = int(b['t0'] * 1e-3 * FS)
            idx_fin = int(b['t1'] * 1e-3 * FS)
            if idx_inicio >= idx_fin: continue
            
            pulso = iq_tensor[:, idx_inicio:idx_fin].clone()
            TARGET_LEN = 131072
            C, L = pulso.shape
            if L < TARGET_LEN:
                pad = torch.zeros(C, TARGET_LEN - L, device=pulso.device)
                pulso_padded = torch.cat([pulso, pad], dim=1)
            else:
                pulso_padded = pulso[:, :TARGET_LEN]
                
            # NUEVO: Generar espectrograma complejo
            input_spec = burst_iq_to_xin_tensor(pulso_padded).unsqueeze(0).to(device)
            
            # 2. CONSTRUIR PERFIL FÍSICO
            feat_array = np.array([
                np.clip(b['dur_ms'], 0, 75), np.clip(abs(b['z_peak']), 0, 30),
                np.clip(b['drop_b'], 0, 10), np.clip(b['n_act'], 0, 2048),
                global_nf, global_ns_val, global_H_mean, global_p75_act
            ], dtype=np.float32)
            
            feat_norm = torch.clamp((torch.from_numpy(feat_array).to(device) - phys_mean) / (phys_std + 1e-8), -5.0, 5.0).unsqueeze(0)
            
            # 3. CLASIFICACIÓN (Pasamos spec y feats al modelo)
            logit = model(input_spec, feat_norm)
            prob_dron = torch.sigmoid(logit).item() * 100 
            
            if prob_dron > 50.0:
                drones_encontrados += 1
                print(f"  [B{i+1:02d}] t={b['t0']:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
            else:
                ruidos_encontrados += 1
                print(f"  [B{i+1:02d}] t={b['t0']:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")
print(f"RESUMEN: {drones_encontrados} Drones | {ruidos_encontrados} Ruidos")

In [ ]:
print("==================================================")
print("  VEREDICTO TEACHER (¿Sabe cazar drones reales?)  ")
print("==================================================")
drones_encontrados = 0
ruidos_encontrados = 0
if len(bursts) == 0:
    print("  ❌ No se detectaron ráfagas. La IA asume que la sala está vacía (RUIDO).")
else:
    # Parámetros Globales estáticos para toda la sala
    global_nf     = float(np.median(nf_v))
    global_ns     = float(np.clip(ns, 0, 5))
    global_H_mean = float(np.mean(H_smooth))
    global_p75_act= float(np.percentile(n_active, 75))
    
    with torch.no_grad():
        for i, b in enumerate(bursts):
            # 1. RECORTAR LA ONDA EXACTA
            idx_inicio = int(b['t0'] * 1e-3 * FS)
            idx_fin = int(b['t1'] * 1e-3 * FS)
            if idx_inicio >= idx_fin:
                continue
            pulso = iq_tensor[:, idx_inicio:idx_fin]
            
            # Normalización RMS de este latido
            power = pulso.pow(2).mean().clamp(min=1e-12).sqrt()
            pulso_normalizado = pulso / power
            
            # --- FIX CRÍTICO DE PADDING ---
            # Rellenar con ceros hasta 131072 muestras (9.4 ms) para simular 
            # el batch padding del entrenamiento y no saturar el AdaptiveAvgPool1d
            TARGET_LEN = 131072
            C, L = pulso_normalizado.shape
            if L < TARGET_LEN:
                pad = torch.zeros(C, TARGET_LEN - L, device=pulso_normalizado.device)
                pulso_padded = torch.cat([pulso_normalizado, pad], dim=1)
            else:
                pulso_padded = pulso_normalizado[:, :TARGET_LEN]
                
            input_ia = pulso_padded.unsqueeze(0).to(device) 
            # ------------------------------
            
            # 2. CONSTRUIR EL PERFIL FÍSICO (8 Dimensiones)
            dur_ms      = np.clip(b['dur_ms'], 0, 75)
            z_peak      = np.clip(abs(b['z_peak']), 0, 30)
            drop_b      = np.clip(b['drop_b'], 0, 10)
            n_act_burst = np.clip(b['n_act'], 0, 2048)
            
            feat_array = np.array([dur_ms, z_peak, drop_b, n_act_burst, 
                                   global_nf, global_ns, global_H_mean, global_p75_act], dtype=np.float32)
            
            feat_t = torch.from_numpy(feat_array).to(device)
            feat_norm = torch.clamp((feat_t - phys_mean) / (phys_std + 1e-8), -5.0, 5.0).unsqueeze(0)
            
            # 3. JUZGADO HÍBRIDO
            logit = model(input_ia, feat_norm)
            prob_dron = torch.sigmoid(logit).item() * 100 
            
            t_ms_inicio = b['t0']
            if prob_dron > 50.0:
                drones_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | z_peak={b['z_peak']:5.1f} | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
            else:
                ruidos_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | z_peak={b['z_peak']:5.1f} | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")


# Prueba pseudo-lábeling para el alumno

In [ ]:
TARGET_DESEADO = 0
SNR_DESEADA    = -12

In [ ]:
archivos_encontrados = glob.glob(os.path.join(
    r"C:\TFM_data\NoisyUAV\drone_RF_data", f"*_target{TARGET_DESEADO}_snr{SNR_DESEADA}.pt"))
if not archivos_encontrados: raise ValueError("❌ No hay muestras.")

semilla = random.randint(0, 9999)
# semilla = 9704
random.seed(semilla)
RUTA_RAW_PARENT = random.choice(archivos_encontrados)
iq_tensor, _, original_target, original_snr = cargar_muestra(RUTA_RAW_PARENT)
print(f"🎬 Muestra: {os.path.basename(RUTA_RAW_PARENT)} (Semilla {semilla})")

FS=14e6; NPERSEG=2048; 
Z_THRESH=2.0; 
MIN_BURST_MS=0.40; 
MERGE_GAP_MS=0.5
MIN_Z_ABS=4.0; 
BG_MULT=4; 
MAX_BINS_FRAC=0.25; 
SMOOTH_MS=0.1; 
ADAPTIVE_WINDOW_MS=15

t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS, min_z_abs=MIN_Z_ABS,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS)

def get_suelo_ms(h_seg, u_seg, dt):
    drop_max = np.max(u_seg - h_seg) + 1e-8
    mask = h_seg < (u_seg - 0.7 * drop_max)
    if not np.any(mask): return 0.0
    return max(len(list(g)) for k, g in itertools.groupby(mask) if k) * dt

print("=" * 70)
print("  ORÁCULO V9 — Plantilla de Duración FHSS (Champion-First)")
print("=" * 70)

if len(bursts) > 0:
    global_nf     = float(np.median(nf_v))
    global_ns     = float(np.clip(ns, 0, 5))
    global_H_mean = float(np.mean(H_smooth))
    global_p75_act= float(np.percentile(n_active, 75))
    dt = float(t_ms[1] - t_ms[0])

    # ── PASE 1: Inferencia IA completa ────────────────────────────────────────
    with torch.no_grad():
        for b in bursts:
            h_seg = H_smooth[b['i0']:b['i1']+1]
            u_seg = umbral_v[b['i0']:b['i1']+1]
            b['suelo_ms']  = get_suelo_ms(h_seg, u_seg, dt)
            b['rugosidad'] = float(np.std(np.diff(h_seg))) if len(h_seg) > 1 else 0.5
            b['label_pseudo'] = -1
            b['rescate'] = None

            idx_i = int(b['t0']*1e-3*FS); idx_f = int(b['t1']*1e-3*FS)
            p_raw  = iq_tensor[:, idx_i:idx_f]
            p_norm = p_raw / p_raw.pow(2).mean().clamp(min=1e-12).sqrt()
            T_LEN  = int(9.4e-3*FS)
            p_pad  = (torch.cat([p_norm, torch.zeros(2, T_LEN-p_norm.shape[1], device=p_norm.device)], dim=1)
                      if p_norm.shape[1] < T_LEN else p_norm[:, :T_LEN])
            feat = torch.from_numpy(np.array([
                np.clip(b['dur_ms'],0,75), np.clip(abs(b['z_peak']),0,30),
                np.clip(b['drop_b'],0,10), np.clip(b['n_act'],0,2048),
                global_nf, global_ns, global_H_mean, global_p75_act
            ], dtype=np.float32)).to(device)
            feat_norm = torch.clamp((feat-phys_mean)/(phys_std+1e-8),-5.,5.).unsqueeze(0)
            b['prob_ia'] = torch.sigmoid(model(p_pad.unsqueeze(0).to(device), feat_norm)).item()*100

    # ── PASE 2: Identificar el CAMPEÓN ───────────────────────────────────────
    # El burst con mayor prob_ia es el más parecido a la firma del dron.
    # Su dur_ms define la plantilla temporal del protocolo FHSS.
    campeon  = max(bursts, key=lambda b: b['prob_ia'])
    ref_dur  = campeon['dur_ms']
    TOL      = 0.15   # ±10% de tolerancia en duración
    print(f"🏆 CAMPEÓN: B{bursts.index(campeon)+1:02d} | "
          f"dur={ref_dur:.2f}ms | n_act={campeon['n_act']:.0f} | "
          f"🧠{campeon['prob_ia']:.1f}% | Tolerancia ±{TOL*100:.0f}%\n")

    # ── PASE 3: Clasificación por plantilla de duración FHSS ─────────────────
    for b in bursts:
        desv_dur = abs(b['dur_ms'] - ref_dur) / (ref_dur + 1e-8)
        dur_ok   = desv_dur <= TOL

        if b is campeon:
            b['label_pseudo'] = 1
            b['rescate'] = "🏆CAMPEÓN"
        elif dur_ok:
            # Misma duración → mismo protocolo FHSS → dron
            b['label_pseudo'] = 1
            b['rescate'] = f"DUR≈ref ({desv_dur*100:.1f}%off)"
        else:
            # Duración diferente → interferencia (BT click, WiFi, AWGN)
            b['label_pseudo'] = 0
            b['rescate'] = f"DUR✗ {b['dur_ms']:.2f}≠{ref_dur:.2f}ms"

    # ── IMPRESIÓN ─────────────────────────────────────────────────────────────
    for i, b in enumerate(bursts):
        tag = "✅ DRON" if b['label_pseudo']==1 else "❌ RUIDO"
        print(f"  [B{i+1:02d}] t={b['t0']:6.2f}ms | dur={b['dur_ms']:.2f}ms | "
              f"suelo={b['suelo_ms']:.2f}ms | rug={b['rugosidad']:.3f} | "
              f"n_act={b['n_act']:.0f} | 🧠{b['prob_ia']:4.1f}% -> {tag} | {b['rescate']}")
else:
    print("  ✗ Sin transmisiones detectadas.")

print("=" * 70)

# ── PLOTTER ───────────────────────────────────────────────────────────────────
fig = plot_muestra(iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
                   fs=FS, titulo="ORÁCULO V9 — Plantilla de Duración FHSS")
for i, b in enumerate(bursts):
    c    = '#27ae60' if b.get('label_pseudo')==1 else '#e74c3c'
    rgba = 'rgba(39,174,96,0.5)' if b.get('label_pseudo')==1 else 'rgba(231,76,60,0.5)'
    for tr in fig.data:
        if isinstance(tr, go.Scatter) and tr.name and f"B{i+1} " in tr.name:
            tr.fillcolor = rgba
    for sh in fig.layout.shapes:
        if sh.type == 'rect' and sh.x0 == b['t0'] and sh.x1 == b['t1']:
            sh.fillcolor = c
    for an in fig.layout.annotations:
        if an.text == f"<b>B{i+1}</b>":
            an.font.color = c
fig.show()
